## Preprocessing

In [147]:
''' preserve word order, keep ! and ? as tokens, and use a minimum document frequency of 2 for the vocabulary'''

import re
from collections import Counter
import numpy as np
import csv

def load_text(filepath):
    texts, labels = [], []
    with open(filepath, 'r', newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            texts.append(row['text'])
            labels.append(int(row['label']))
    return texts, labels

train_texts, train_labels = load_text('train_data.csv')
val_texts, val_labels = load_text('val_data.csv')
test_texts, test_labels = load_text('test_data.csv')


token_pattern = re.compile(r"[\w!?]+") #keep the ! and ? just like before

def tokenize(text):
    return token_pattern.findall(text)

def build_vocab(texts, min_df=2):
    token_counts = Counter()
    for text in texts:
        tokens = tokenize(text)
        token_counts.update(tokens)
    return {token for token, count in token_counts.items() if count >= min_df}  #return tokens that appear at least min_df times matching other model

vocab = build_vocab(train_texts, min_df=2)  

word2idx = {'<PAD>': 0, '<UNK>': 1}  # reserve index 0 for padding and 1 for unknown tokens
start_idx = 2
for word in vocab:
    word2idx[word] = start_idx
    start_idx += 1              #assign index to each word in vocab starting with 2





def tokens_index(tokens, word2idx):
    return [word2idx.get(token, word2idx['<UNK>']) for token in tokens]  #look up the index of each token in word2idx, if not found return index of <UNK>


train_tokenized = [tokens_index(tokenize(text), word2idx) for text in train_texts]
val_tokenized = [tokens_index(tokenize(text), word2idx) for text in val_texts]
test_tokenized = [tokens_index(tokenize(text), word2idx) for text in test_texts]

maxlen = max(len(tokens) for tokens in train_tokenized)  #find the maximum length of tokens in training data, are similar lengths already 

def pad_sequence(ids, maxlen, pad_id=0):
    if len(ids) >= maxlen:
        return ids[:maxlen]
    return ids + [pad_id] * (maxlen - len(ids))

train_padded = np.array([pad_sequence(ids, maxlen) for ids in train_tokenized])
train_mask = (train_padded !=0).astype(int) #create an array indicating 1 for real token vs padding 0 will use later


val_padded = np.array([pad_sequence(ids, maxlen) for ids in val_tokenized])
val_mask = (val_padded != 0).astype(int)

#reshape for training
y_train = np.array(train_labels).reshape(-1, 1)
y_val = np.array(val_labels).reshape(-1, 1)


vocab_size = len(word2idx)  




In [ ]:


def init_embeddings(vocab_size, embedding_dims=50):
    scale = np.sqrt(1.0 / embedding_dims)  #inital size was too small
    embedding_matrix = np.random.randn(vocab_size, embedding_dims) * scale
    return embedding_matrix

def embedding_forward(token_ids, embedding_matrix):
    embedded = embedding_matrix[token_ids] #match the token ids to our embedding matrix
    return embedded

def embedding_backward(d_embedded, token_ids, vocab_size, embedding_dims):  
    d_embedding_matrix = np.zeros((vocab_size, embedding_dims)) 
    np.add.at(d_embedding_matrix, token_ids, d_embedded) #allows us to process multiple identical ids in the same document wihtout overwriting
    return d_embedding_matrix


In [149]:
def init_weights(d_k, embedding_dims=50):
   # using as initial setting wouldn't work
    scale = np.sqrt(1.0 / embedding_dim)

    wq = np.random.randn(embedding_dim, d_k) * scale
    wk = np.random.randn(embedding_dim, d_k) * scale
    wv = np.random.randn(embedding_dim, d_k) * scale
    return wq, wk, wv

def softmax(x, axis=-1):
    x_stable = x - np.max(x, axis=axis, keepdims=True) #for numerical stability precent overflow
    exp_x = np.exp(x_stable)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

def forward_pass_att(embedded, mask, wq, wk, wv, d_k):
    Q, K, V = embedded @ wq, embedded @ wk, embedded @ wv
    scores = (Q @ K.transpose(0, 2, 1)) / np.sqrt(d_k)
    
    exp_scores = np.exp(scores - np.max(scores, axis=-1, keepdims=True)) # softmax across sequence dimension
    attn_weights = exp_scores / np.sum(exp_scores, axis=-1, keepdims=True)

    attention_output = (attn_weights @ V) * mask[:, :, None]
    
    return attention_output, attn_weights, Q, K, V




## Pooling and backprop for attention

In [ ]:
import numpy as np


def masked_pool(attention_output, mask):
    # zero out pad tokens
    mask_expanded = mask[:, :, None]
    masked_output = attention_output * mask_expanded
    
    # average across valid tokens
    summed_output = masked_output.sum(axis=1)
    counts = mask.sum(axis=1, keepdims=True)
    pooled = summed_output / counts

    return pooled, counts


def back_pool(d_pool, mask, counts):
    d_sum = d_pool / counts
    d_masked = d_sum[:, None, :]
    d_attention_output = d_masked * mask[:, :, None]

    return d_attention_output

def attention_backward(d_attn_out, Q, K, V, attn_weights, mask, emb, wq, wk, wv, d_k):
    batch_size = emb.shape[0]
    
    
    d_V = attn_weights.transpose(0, 2, 1) @ d_attn_out
    d_attn_weights = d_attn_out @ V.transpose(0, 2, 1)
    
    
    sum_d_attn = np.sum(d_attn_weights * attn_weights, axis=-1, keepdims=True)
    d_scores = attn_weights * (d_attn_weights - sum_d_attn)
    d_scores = d_scores / np.sqrt(d_k)
    
 
    d_Q = d_scores @ K
    d_K = d_scores.transpose(0, 2, 1) @ Q
    

    d_wq = np.sum(emb.transpose(0, 2, 1) @ d_Q, axis=0) / batch_size
    d_wk = np.sum(emb.transpose(0, 2, 1) @ d_K, axis=0) / batch_size
    d_wv = np.sum(emb.transpose(0, 2, 1) @ d_V, axis=0) / batch_size
    
    
    d_emb = (d_Q @ wq.T) + (d_K @ wk.T) + (d_V @ wv.T)
    
    return d_emb, d_wq, d_wk, d_wv


In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -500, 500)))

# simple wrapper to run forward pass end-to-end
def forward_pass(token_ids, mask, embedding_matrix, wq, wk, wv, d_k, w_out, b_out):
    emb = embedding_forward(token_ids, embedding_matrix)
    attn_out, attn_weights, Q, K, V = forward_pass_att(emb, mask, wq, wk, wv, d_k) 
                                               
    pooled, counts = masked_pool(attn_out, mask) # pool seq vectors and project to output score
    logits = pooled @ w_out + b_out
    preds = sigmoid(logits)
    
    
    cache = (token_ids, emb, Q, K, V, attn_weights, mask, counts, pooled, preds, d_k)  # save for backward
    return preds, cache


def backward_pass(d_loss, cache, embedding_matrix, wq, wk, wv, w_out):
    token_ids, emb, Q, K, V, attn_weights, mask, counts, pooled, preds, d_k = cache
    batch_size = token_ids.shape[0]
    
    # Average logit gradient over mini-batch
    d_logits = d_loss / batch_size
    
    d_w_out = pooled.T @ d_logits
    d_b_out = np.sum(d_logits, axis=0, keepdims=True)
    d_pooled = d_logits @ w_out.T
    
    # pool to attention, attention to embeddings
    d_attn_out = back_pool(d_pooled, mask, counts)
    d_emb, d_wq, d_wk, d_wv = attention_backward(d_attn_out, Q, K, V, attn_weights, mask, emb, wq, wk, wv, d_k)
    
    v_size, e_dim = embedding_matrix.shape
    d_emb_matrix = embedding_backward(d_emb, token_ids, v_size, e_dim)
    
    grads = {
        'embedding_matrix': d_emb_matrix,
        'wq': d_wq,
        'wk': d_wk,
        'wv': d_wv,
        'w_out': d_w_out,
        'b_out': d_b_out
    }
    return grads

def update_weights(params, grads, lr=0.01):
    for k in params:
        params[k] -= lr * grads[k]
    return params

def compute_bce_loss(preds, targets):
    eps = 1e-15
    preds_clipped = np.clip(preds, eps, 1 - eps)
    loss = -np.mean(targets * np.log(preds_clipped) + (1 - targets) * np.log(1 - preds_clipped))
    return loss

def compute_bce_grad(preds, targets):
     return preds - targets 

def create_mini_batches(X, mask, y, batch_size=32, shuffle=True):
    n_samples = X.shape[0]
    indices = np.arange(n_samples)
    if shuffle:
        np.random.shuffle(indices)
        
    for start_idx in range(0, n_samples, batch_size):
        batch_idx = indices[start_idx : start_idx + batch_size]
        # reshape y to avoid bugs
        yield X[batch_idx], mask[batch_idx], y[batch_idx].reshape(-1, 1)

In [152]:
class SimpleAttentionClassifier:
    def __init__(self, vocab_size, embedding_dim, d_k):
        embedding_matrix = init_embeddings(vocab_size, embedding_dim)
        wq, wk, wv = init_weights(d_k, embedding_dims=embedding_dim)
        w_out = np.random.randn(embedding_dim, 1) * 0.01
        b_out = np.zeros((1, 1))
        self.params = {
            'embedding_matrix': embedding_matrix,
            'wq': wq, 'wk': wk, 'wv': wv,
            'w_out': w_out, 'b_out': b_out
        }
        self.d_k = d_k
    
    def train(self, x_train, mask_train, y_train, x_val, mask_val, y_val, epochs=100, learning_rate=0.01):
        print("Epochs:", epochs)
        history = {'train_loss': [], 'val_loss': [], 'val_accuracy': [], 'train_accuracy': []}

        for epoch in range(epochs):
            mini_batches = create_mini_batches(x_train, mask_train, y_train, batch_size=32)
            epoch_loss = 0
            num_batches = 0  

            for x_batch, mask_batch, y_batch in mini_batches:
                output, cache = forward_pass(
                    x_batch, mask_batch, 
                    self.params['embedding_matrix'], self.params['wq'], self.params['wk'], self.params['wv'], 
                    self.d_k, self.params['w_out'], self.params['b_out']
                )

                loss = compute_bce_loss(output, y_batch)
                d_loss = compute_bce_grad(output, y_batch)

                grads = backward_pass(
                    d_loss, cache, 
                    self.params['embedding_matrix'], self.params['wq'], self.params['wk'], self.params['wv'], 
                    self.params['w_out']
                )

                self.params = update_weights(self.params, grads, lr=learning_rate)
                epoch_loss += loss
                num_batches += 1  

            average_train_loss = epoch_loss / num_batches  
            train_output, _ = forward_pass(
                x_train, mask_train, 
                self.params['embedding_matrix'], self.params['wq'], self.params['wk'], self.params['wv'], 
                self.d_k, self.params['w_out'], self.params['b_out']
            )
        
            train_accuracy = np.mean((train_output > 0.5) == y_train)
            
            val_output, _ = forward_pass(
                x_val, mask_val, 
                self.params['embedding_matrix'], self.params['wq'], self.params['wk'], self.params['wv'], 
                self.d_k, self.params['w_out'], self.params['b_out']
            )
            val_loss = compute_bce_loss(val_output, y_val)
            val_accuracy = np.mean((val_output > 0.5) == y_val)

            history['train_accuracy'].append(train_accuracy)
            history['val_accuracy'].append(val_accuracy)
            history['train_loss'].append(average_train_loss)
            history['val_loss'].append(val_loss)

            if epoch % 10 == 0 or epoch == epochs - 1:
                print(f"Epoch {epoch}, Train Loss: {average_train_loss:.4f}, Val Loss: {val_loss:.4f}")

            if epoch == epochs - 1:
                best_epoch = np.argmin(history['val_loss'])
                print(f"Best epoch: {best_epoch}, Best Val Loss: {history['val_loss'][best_epoch]:.4f}, "
                      f"Train Accuracy: {history['train_accuracy'][best_epoch]:.4f}, "
                      f"Val Accuracy: {history['val_accuracy'][best_epoch]:.4f}")

        return history

In [153]:
# hyperparameters
embedding_dim = 50
d_k = 50


model = SimpleAttentionClassifier(vocab_size=vocab_size, embedding_dim=embedding_dim, d_k=d_k)


history = model.train(
    x_train=train_padded,
    mask_train=train_mask,
    y_train=y_train,
    x_val=val_padded,
    mask_val=val_mask,
    y_val=y_val,
    epochs=10,
    learning_rate=0.08
)

Epochs: 10
Epoch 0, Train Loss: 0.6934, Val Loss: 0.6931
Epoch 9, Train Loss: 0.6933, Val Loss: 0.6931
Best epoch: 9, Best Val Loss: 0.6931, Train Accuracy: 0.5337, Val Accuracy: 0.5425


In [154]:
print("len(word2idx):", len(word2idx))
print("Current vocab_size variable:", vocab_size)

len(word2idx): 8628
Current vocab_size variable: 8628


In [155]:
# 1. Grab a single mini-batch
x_b, m_b, y_b = train_padded[:32], train_mask[:32], y_train[:32]

# 2. Run Forward Pass
out, cache = forward_pass(
    x_b, m_b, 
    model.params['embedding_matrix'], model.params['wq'], model.params['wk'], model.params['wv'], 
    model.d_k, model.params['w_out'], model.params['b_out']
)

# 3. Run Backward Pass
d_loss = compute_bce_grad(out, y_b)
grads = backward_pass(
    d_loss, cache, 
    model.params['embedding_matrix'], model.params['wq'], model.params['wk'], model.params['wv'], 
    model.params['w_out']
)

# 4. Print Gradient Norms
for key, value in grads.items():
    print(f"Gradient norm for {key:18s}: {np.linalg.norm(value):.8f}")

Gradient norm for embedding_matrix  : 0.00783669
Gradient norm for wq                : 0.00000000
Gradient norm for wk                : 0.00000000
Gradient norm for wv                : 0.00001833
Gradient norm for w_out             : 0.00698669
Gradient norm for b_out             : 0.06226864
